# Compare OWLv2 Thresholds: SAM2 vs SAM-HQ

Randomly sample 10 frames and compare OWLv2 boxes at thresholds 0.12, 0.15, and 0.18 using two mask generators:

- Row 1: Original | OWLv2+SAM2 0.12 | OWLv2+SAM2 0.15 | OWLv2+SAM2 0.18
- Row 2: Original | OWLv2+SAM-HQ 0.12 | OWLv2+SAM-HQ 0.15 | OWLv2+SAM-HQ 0.18

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import random
import sys
import time
import traceback
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None
    DEVICE = "cpu"


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "scripts" / "auto_label" / "auto_label_gui.py").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find repo root. Start Jupyter from prompt_video_segmenter or set REPO_ROOT manually.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

FRAME_DIR = REPO_ROOT / "data" / "auto_label_demo" / "frames"
OUTPUT_DIR = REPO_ROOT / "outputs" / "compare_owlv2_sam2_samhq_thresholds_10frames"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_SAMPLES = 10
THRESHOLDS = [0.12, 0.15, 0.18]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Repo root : {REPO_ROOT}")
print(f"Frame dir : {FRAME_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device    : {DEVICE}")

## 2. Locate frames and sample 10 images

In [ ]:
def list_images(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


all_frames = list_images(FRAME_DIR)
if len(all_frames) < N_SAMPLES:
    raise ValueError(f"Only found {len(all_frames)} images in {FRAME_DIR}; need {N_SAMPLES}.")

rng = random.Random(RANDOM_SEED)
sampled_frames = rng.sample(all_frames, N_SAMPLES)

print(f"Found {len(all_frames)} frames; sampled {len(sampled_frames)}.")
for i, path in enumerate(sampled_frames):
    print(f"{i:02d}: {path}")

## 3. Load text prompts from existing autolabel GUI

In [ ]:
# Copied from AutoLabelWindow._build_config_panel() in scripts/auto_label/auto_label_gui.py.
TEXT_PROMPTS = [
    "hand", "glove",
    "pot", "pan", "lid", "cookware", "tray", "kettle",
    "bowl", "plate", "cup", "glass", "bottle", "jar",
    "container", "box", "package", "bag", "carton", "can",
    "knife", "fork", "spoon", "spatula", "tongs", "ladle", "whisk", "peeler", "scissors", "cutting board",
    "pasta", "noodles", "rice", "bread", "vegetable", "fruit", "meat", "fish", "egg", "cheese",
    "ingredient", "food", "dry food", "liquid", "water", "milk", "sauce", "oil", "powder", "sugar", "salt",
    "sink", "faucet", "stove", "cooktop", "oven", "microwave", "fridge",
    "drawer", "cabinet", "countertop", "table", "rack", "sponge", "towel",
]

print(f"Loaded {len(TEXT_PROMPTS)} prompts:")
print(", ".join(TEXT_PROMPTS))

## 4. Load OWLv2 + SAM2 and OWLv2 + SAM-HQ

In [ ]:
import owlv2_sam2_adapter_for_compare as sam2_adapter
import owlv2_samhq_adapter_for_compare as samhq_adapter

sam2_state = None
samhq_state = None
sam2_load_error = ""
samhq_load_error = ""

try:
    sam2_state = sam2_adapter.load_sam3_model(DEVICE)
    print(f"OWLv2+SAM2 loaded. SAM2 available={sam2_state['sam2_available']} {sam2_state['sam2_error']}")
except Exception as exc:
    sam2_load_error = f"{type(exc).__name__}: {exc}"
    print(f"OWLv2+SAM2 failed: {sam2_load_error}")
    traceback.print_exc(limit=2)

try:
    samhq_state = samhq_adapter.load_owlv2_samhq(DEVICE)
    print("OWLv2+SAM-HQ loaded.")
except Exception as exc:
    samhq_load_error = f"{type(exc).__name__}: {exc}"
    print(f"OWLv2+SAM-HQ failed: {samhq_load_error}")
    traceback.print_exc(limit=2)

## 5. Run inference

In [ ]:
def run_sam2(image_path: Path, threshold: float) -> dict:
    started = time.perf_counter()
    if sam2_state is None:
        return {"instances": [], "error": sam2_load_error or "SAM2 state missing", "seconds": time.perf_counter() - started}
    try:
        instances = sam2_adapter.predict_owlv2_sam2(sam2_state, image_path, TEXT_PROMPTS, score_threshold=threshold)
        return {"instances": instances, "error": "", "seconds": time.perf_counter() - started}
    except Exception as exc:
        return {"instances": [], "error": f"{type(exc).__name__}: {exc}", "seconds": time.perf_counter() - started}


def run_samhq(image_path: Path, threshold: float) -> dict:
    started = time.perf_counter()
    if samhq_state is None:
        return {"instances": [], "error": samhq_load_error or "SAM-HQ state missing", "seconds": time.perf_counter() - started}
    try:
        instances = samhq_adapter.predict_owlv2_samhq(samhq_state, image_path, TEXT_PROMPTS, score_threshold=threshold)
        return {"instances": instances, "error": "", "seconds": time.perf_counter() - started}
    except Exception as exc:
        return {"instances": [], "error": f"{type(exc).__name__}: {exc}", "seconds": time.perf_counter() - started}


results = []
for idx, image_path in enumerate(sampled_frames):
    print(f"[{idx + 1:02d}/{len(sampled_frames)}] {image_path}")
    row = {"frame_path": image_path, "sam2": {}, "samhq": {}}
    for threshold in THRESHOLDS:
        row["sam2"][threshold] = run_sam2(image_path, threshold)
        print(f"  SAM2  {threshold:.2f}: {len(row['sam2'][threshold]['instances'])} instances, {row['sam2'][threshold]['seconds']:.2f}s")
    for threshold in THRESHOLDS:
        row["samhq"][threshold] = run_samhq(image_path, threshold)
        print(f"  SAMHQ {threshold:.2f}: {len(row['samhq'][threshold]['instances'])} instances, {row['samhq'][threshold]['seconds']:.2f}s")
    results.append(row)

print("Done.")

## 6. Visualize 2x4 comparisons

In [ ]:
def load_rgb(image_path: Path) -> np.ndarray:
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def color_for_index(index: int) -> tuple[float, float, float]:
    cmap = plt.get_cmap("tab20")
    return tuple(cmap(index % 20)[:3])


def normalize_instances(instances: list[dict]) -> list[dict]:
    rows = []
    for inst in instances or []:
        rows.append({
            "label": inst.get("label", inst.get("class", inst.get("name", "object"))),
            "score": inst.get("confidence", inst.get("score", None)),
            "bbox": inst.get("bbox_xyxy", inst.get("bbox", None)),
            "mask": inst.get("mask", None),
        })
    return rows


def draw_result(ax, image_rgb: np.ndarray, instances: list[dict], title: str, error: str = "") -> None:
    ax.imshow(image_rgb)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    if error:
        ax.text(8, 22, error[:90], color="white", fontsize=7, bbox={"facecolor": "red", "alpha": 0.7})
        return

    for i, inst in enumerate(normalize_instances(instances)):
        color = color_for_index(i)
        mask = inst.get("mask")
        if mask is not None:
            mask_arr = np.asarray(mask).astype(bool)
            if mask_arr.shape[:2] == image_rgb.shape[:2]:
                overlay = np.zeros((*mask_arr.shape, 4), dtype=float)
                overlay[mask_arr, :3] = color
                overlay[mask_arr, 3] = 0.35
                ax.imshow(overlay)

        bbox = inst.get("bbox")
        if bbox is not None and len(bbox) >= 4:
            x1, y1, x2, y2 = [float(v) for v in bbox[:4]]
            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=1.2, edgecolor=color)
            ax.add_patch(rect)
            label = str(inst.get("label", "object"))
            score = inst.get("score")
            if score is not None:
                label = f"{label} {float(score):.2f}"
            ax.text(x1, max(0, y1 - 3), label, color="white", fontsize=6, bbox={"facecolor": color, "alpha": 0.8, "pad": 1})


comparison_paths = []
for idx, row in enumerate(results):
    image_rgb = load_rgb(row["frame_path"])
    fig, axes = plt.subplots(2, 4, figsize=(18, 9), constrained_layout=True)

    draw_result(axes[0, 0], image_rgb, [], "Original")
    for col, threshold in enumerate(THRESHOLDS, start=1):
        result = row["sam2"][threshold]
        draw_result(axes[0, col], image_rgb, result["instances"], f"OWLv2+SAM2 {threshold:.2f}\nN={len(result['instances'])}", result.get("error", ""))

    draw_result(axes[1, 0], image_rgb, [], "Original")
    for col, threshold in enumerate(THRESHOLDS, start=1):
        result = row["samhq"][threshold]
        draw_result(axes[1, col], image_rgb, result["instances"], f"OWLv2+SAM-HQ {threshold:.2f}\nN={len(result['instances'])}", result.get("error", ""))

    out_path = OUTPUT_DIR / f"owlv2_sam2_samhq_{idx:03d}.png"
    fig.savefig(out_path, dpi=160)
    comparison_paths.append(out_path)
    plt.show()

print(f"Saved {len(comparison_paths)} figures to {OUTPUT_DIR}")

## 7. Save summary CSV

In [ ]:
summary_rows = []
for row, out_path in zip(results, comparison_paths):
    record = {
        "frame_path": str(row["frame_path"]),
        "comparison_output_path": str(out_path),
    }
    for threshold in THRESHOLDS:
        sam2_result = row["sam2"][threshold]
        samhq_result = row["samhq"][threshold]
        record[f"sam2_{threshold:.2f}_num_instances"] = len(sam2_result["instances"])
        record[f"sam2_{threshold:.2f}_num_masks"] = sum(1 for x in sam2_result["instances"] if x.get("mask") is not None)
        record[f"sam2_{threshold:.2f}_seconds"] = sam2_result.get("seconds", None)
        record[f"sam2_{threshold:.2f}_error"] = sam2_result.get("error", "")
        record[f"samhq_{threshold:.2f}_num_instances"] = len(samhq_result["instances"])
        record[f"samhq_{threshold:.2f}_num_masks"] = sum(1 for x in samhq_result["instances"] if x.get("mask") is not None)
        record[f"samhq_{threshold:.2f}_seconds"] = samhq_result.get("seconds", None)
        record[f"samhq_{threshold:.2f}_error"] = samhq_result.get("error", "")
    summary_rows.append(record)

summary_df = pd.DataFrame(summary_rows)
summary_csv = OUTPUT_DIR / "summary.csv"
summary_df.to_csv(summary_csv, index=False)
display(summary_df)
print(f"Saved: {summary_csv}")

timing_cols = [c for c in summary_df.columns if c.endswith("_seconds")]
if timing_cols:
    print("Average seconds per frame/threshold:")
    display(summary_df[timing_cols].mean().to_frame("mean_seconds").T)
